# Notebook 04: Model Evaluation, Comparison & Interpretability
## Project: Online Shopping Purchase Intention Prediction
**University AI & Machine Learning Group Assignment**

This notebook evaluates the tuned SVM and Decision Tree models on the **untouched 20% test dataset** (2,466 sessions). We compare performance metrics, confusion matrices, ROC and Precision-Recall curves, and analyze feature importance.


In [ ]:
import json
import pandas as pd
import numpy as np

# Load actual model results generated from the genuine pipeline execution
with open('../backend/artifacts/metrics/model_results.json') as f:
    results = json.load(f)

comp_df = pd.DataFrame(results['comparison_table'])
print("=== Quantitative Performance Comparison (Untouched Test Set) ===")
comp_df


### 1. Classification Reports
Examining class-specific Precision, Recall, and F1-Scores for both algorithms.


In [ ]:
for model_name in ['SVM', 'Decision Tree']:
    print(f"--- {model_name} Classification Report ---")
    cr = results['models'][model_name]['classification_report']
    print(pd.DataFrame(cr).T.round(4))
    print()


### 2. Confusion Matrix Analysis
- True Negatives (TN): Correctly identified non-purchasers
- False Positives (FP): Non-purchasers incorrectly flagged as prospective buyers (false alarm)
- False Negatives (FN): Missed buyers who were about to purchase (missed revenue opportunity)
- True Positives (TP): Accurately identified buyers


In [ ]:
svm_cm = np.array(results['models']['SVM']['confusion_matrix'])
dt_cm = np.array(results['models']['Decision Tree']['confusion_matrix'])

print(f"SVM Test Confusion Matrix:\n{svm_cm}\n")
print(f"Decision Tree Test Confusion Matrix:\n{dt_cm}")


### 3. Feature Importance Analysis
- **Decision Tree**: Native Gini/Entropy Impurity Reduction across derived one-hot encoded features.
- **SVM**: Permutation Importance measuring drop in test F1 when feature values are permuted.


In [ ]:
print("Top 5 Decision Tree Features:")
print(pd.DataFrame(results['feature_importance']['decision_tree_feature_importance'][:5]))

print("\nTop 5 SVM Permutation Features:")
print(pd.DataFrame(results['feature_importance']['svm_permutation_importance'][:5]))


### Final Analytical Conclusion:
1. **Model Selection**: SVM achieves higher overall predictive balance (Test F1 = 0.6419, CV Mean F1 = 0.6655, Accuracy = 87.47%, ROC-AUC = 0.8997) compared to the Decision Tree (Test F1 = 0.6285, CV Mean F1 = 0.6344).
2. **Recall vs Precision Trade-off**: The Decision Tree captures higher Recall (81.94% vs 72.51%) but suffers from lower Precision (50.98% vs 57.59%), generating 301 false positives compared to SVM's 204.
3. **Primary Predictive Drivers**: `PageValues` is unequivocally the most decisive predictor of purchase conversion, followed by seasonality (`Month_Nov`) and visitor browsing duration.
